<a href="https://colab.research.google.com/github/arjunabde-coder/Brain-Tumor-Detector-Project/blob/main/braintumor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install split-folders scikit-learn tensorflow matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, shutil
import cv2
import time
import zipfile
import tensorflow as tf
import imutils
import splitfolders
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import Sequential
import matplotlib.image as mpimg
import seaborn as sns
%matplotlib inline
plt.style.use('ggplot')

In [ ]:
zip_file = "/content/archive.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("Brain Tumor Dataset")

In [ ]:
base_path = "/content/Brain Tumor Dataset/Training"
tumor_types = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

def count_files(base_path):
    for tumor_type in tumor_types:
        folder_path = os.path.join(base_path, tumor_type)
        if os.path.exists(folder_path):
            file_list = os.listdir(folder_path)
            print(f"Number of files in {tumor_type}: {len(file_list)}")
        else:
            print(f"{tumor_type} folder not found!")

print("Brain Tumor MRI Dataset EDA")
count_files(base_path)

In [ ]:
def timing(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int(sec_elapsed % (60 * 60) / 60)
    s = sec_elapsed % 60
    return f"{h}:{m}:{s}"

def augmented_data(base_dir, n_generated_samples, save_base_dir):
    data_gen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        brightness_range=(0.3, 1.0),
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest'
    )
    start_time = time.time()

    for tumor_type in tumor_types:
        source_dir = os.path.join(base_dir, tumor_type)
        save_dir = os.path.join(save_base_dir, tumor_type)
        os.makedirs(save_dir, exist_ok=True)

        for filename in os.listdir(source_dir):
            image = cv2.imread(os.path.join(source_dir, filename))
            image = image.reshape((1,) + image.shape)
            save_prefix = 'aug_' + filename[:-4]
            i = 0

            for batch in data_gen.flow(x=image, batch_size=1, save_to_dir=save_dir, save_prefix=save_prefix, save_format="jpg"):
                i += 1
                if i >= n_generated_samples:
                    break

    end_time = time.time()
    print(f"Data augmentation completed in: {timing(end_time - start_time)}")

base_dir = '/content/Brain Tumor Dataset/Training'
n_generated_samples = 5
save_base_dir = '/content/Augmented Data'
augmented_data(base_dir, n_generated_samples, save_base_dir)


In [ ]:
def rename_files(folder, tumor_name):
    count = 1
    for filename in os.listdir(folder):
        source = os.path.join(folder, filename)
        destination = os.path.join(folder, f"{tumor_name}_{count}.jpg")
        os.rename(source, destination)
        count += 1
    print(f"{tumor_name} Tumor Images Renamed Successfully.")

for tumor in tumor_types:
    folder = f'/content/Augmented Data/{tumor}/'
    rename_files(folder, tumor)

In [ ]:
tumor_counts = [len(os.listdir(os.path.join(save_base_dir, tumor))) for tumor in tumor_types]
print("Tumor type counts (Augmented Data):", tumor_counts)

colors = ['#006A67', '#872341', '#1D1616', '#4B4376']
x = np.arange(len(tumor_types))
width = 0.4

plt.figure(figsize=(10, 6))
plt.bar(x, tumor_counts, width=width, align='center', alpha=0.7, color=colors)
plt.xlabel('Tumor Types')
plt.ylabel('Number of Images')
plt.title('Number of Augmented Images per Tumor Type')
plt.xticks(x, tumor_types, rotation=45)
plt.show()

In [ ]:
def crop_brain_tumor(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    thres = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thres = cv2.erode(thres, None, iterations=2)
    thres = cv2.dilate(thres, None, iterations=2)
    cnts = cv2.findContours(thres.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)

    if len(cnts) == 0:
        return image

    c = max(cnts, key=cv2.contourArea)
    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])

    cropped_image = image[extTop[1]:extBot[1], extLeft[0]:extRight[0]]

    return cropped_image

def crop_and_save_images(folder_path):
    for filename in os.listdir(folder_path):
        image_path = os.path.join(folder_path, filename)
        image = cv2.imread(image_path)

        if image is not None:
            cropped_image = crop_brain_tumor(image)
            cv2.imwrite(image_path, cropped_image)
            print(f"Processed: {filename}")
        else:
            print(f"Failed to read: {filename}")

for tumor in tumor_types:
    folder_path = os.path.join(save_base_dir, tumor)
    print(f"Processing folder: {folder_path}")
    crop_and_save_images(folder_path)

In [ ]:
input_folder = "/content/Augmented Data"
output_folder = "/content/Split Data"

splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(0.8, 0.1, 0.1))
print("Dataset successfully split into training, validation, and test sets.")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('Split_Data', 'zip', 'Split Data')

files.download('Split_Data.zip')


In [ ]:
def build_cnn_model():
    model = Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4, activation='softmax')  # 4 classes
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_cnn_model()
model.summary()

In [ ]:
train_dir = "/content/Split Data/train"
val_dir = "/content/Split Data/val"
test_dir = "/content/Split Data/test"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
checkpoint = tf.keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True)

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

model = tf.keras.models.load_model('/content/best_model.h5')
class_names = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img, img_array

def predict_image(img_path):
    img, img_array = load_and_preprocess_image(img_path)
    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    print(f"Predicted Tumor Type: {class_names[predicted_class[0]]}")

img_path = '/content/Split Data/test/pituitary_tumor/pituitary_tumor_1040.jpg'
predict_image(img_path)